# Scenario 1 - Pyspark

In [0]:
from pyspark.sql import SparkSession

# Create Spark Session
spark = SparkSession.builder.appName("WorkerDataFrame").getOrCreate()

# Sample Data
data = [
    ("001", "Monika", "Arora", 100000, "2014-02-20 09:00:00", "HR"),
    ("002", "Niharika", "Verma", 300000, "2014-06-11 09:00:00", "Admin"),
    ("003", "Vishal", "Singhal", 300000, "2014-02-20 09:00:00", "HR"),
    ("004", "Amitabh", "Singh", 500000, "2014-02-20 09:00:00", "Admin"),
    ("005", "Vivek", "Bhati", 500000, "2014-06-11 09:00:00", "Admin")
]

# Column Names
columns = [
    "workerid",
    "firstname",
    "lastname",
    "salary",
    "joiningdate",
    "depart"
]

# Create DataFrame
df = spark.createDataFrame(data, columns)
df.printSchema()

# Display DataFrame
df.show(truncate=False)
#
# Convert joiningdate to TimestampType
from pyspark.sql.functions import col
df = df.withColumn("joiningdate", col("joiningdate").cast("timestamp"))


# Display DataFrame
df.show(truncate=False)

# Print Schema
df.printSchema()


In [0]:
from pyspark.sql.functions import col

# Find workers with equal salary
equal_salary_df = df.join(
    df.groupBy("salary").count().filter(col("count") > 1).select("salary"),
    on="salary",
    how="inner"
)

display(equal_salary_df)

In [0]:


finaldf = df.alias("a").join(
    df.alias("b"),
    (col("a.salary") == col("b.salary")) & (col("a.workerid") != col("b.workerid"))
).select("a.*")
finaldf.show()

# **SQL**


In [0]:
%sql
CREATE TABLE worker (
    workerid   INT PRIMARY KEY,
    firstname  VARCHAR(50),
    lastname   VARCHAR(50),
    salary     INT,
    joiningdate TIMESTAMP,
    depart     VARCHAR(50)
);

-- Insert Rows
INSERT INTO worker (workerid, firstname, lastname, salary, joiningdate, depart) VALUES
(001, 'Monika',   'Arora',   100000, '2014-02-20 09:00:00', 'HR'),
(002, 'Niharika', 'Verma',   300000, '2014-06-11 09:00:00', 'Admin'),
(003, 'Vishal',   'Singhal', 300000, '2014-02-20 09:00:00', 'HR'),
(004, 'Amitabh',  'Singh',   500000, '2014-02-20 09:00:00', 'Admin'),
(005, 'Vivek',    'Bhati',   500000, '2014-06-11 09:00:00', 'Admin');

-- Verify
SELECT * FROM worker;



In [0]:
%sql
select a.workerid, a.firstname, a.salary
from worker a
inner join worker b 
on a.salary = b.salary
where a.workerid != b.workerid;


# Scenario 2 - Pyspark

In [0]:
data = [
    (1, "1-Jan", "Ordered"),
    (1, "2-Jan", "dispatched"),
    (1, "3-Jan", "dispatched"),
    (1, "4-Jan", "Shipped"),
    (1, "5-Jan", "Shipped"),
    (1, "6-Jan", "Delivered"),
    (2, "1-Jan", "Ordered"),
    (2, "2-Jan", "dispatched"),
    (2, "3-Jan", "shipped"),
]

columns = ["orderid", "statusdate", "status"]
df = spark.createDataFrame(data, columns)


from pyspark.sql.window import Window
from pyspark.sql.functions import lag, col

# Define window partitioned by orderid, ordered by status date
window = Window.partitionBy("orderid").orderBy("statusdate")

# Add previous status column using LAG
df_lag = df.withColumn("pre_status", lag("status").over(window))
#df_lag.show()

# Filter: keep first row OR rows where status changed
result_df = df_lag.filter(
    col("pre_status").isNull() |
    (col("status") != col("pre_status"))
).select("orderid", "statusdate", "status")

result_df.show()


In [0]:
%sql
CREATE TABLE orders (
    orderid    INT,
    statusdate VARCHAR(10),
    status     VARCHAR(20)
);

-- Insert Rows
INSERT INTO orders VALUES
(1, '1-Jan', 'Ordered'),
(1, '2-Jan', 'dispatched'),
(1, '3-Jan', 'dispatched'),
(1, '4-Jan', 'Shipped'),
(1, '5-Jan', 'Shipped'),
(1, '6-Jan', 'Delivered'),
(2, '1-Jan', 'Ordered'),
(2, '2-Jan', 'dispatched'),
(2, '3-Jan', 'shipped');



In [0]:
%sql
select * from orders;

-- Query: Get dates when status changes
select orderid, statusdate, status
from( 
    select *,
        lag(status) over(partition by orderid order by statusdate) as prev_status
    from orders
)t
where prev_status is null 
   or status != prev_status;


/*
Get dates when status changes for each order:
- First record of each order (prev_status IS NULL)
- Status changed from previous row (status != prev_status)

SELECT orderid, statusdate, status
FROM (
    SELECT *,
           LAG(status) OVER (PARTITION BY orderid ORDER BY statusdate) AS prev_status
    FROM orders
) t
WHERE prev_status IS NULL
   OR status != prev_status;

*/